In [78]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import re
import warnings
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')


In [79]:
PALETTE   = ["#1A3C5E","#2E86AB","#4ECDC4","#FF6B6B","#FFA500",
             "#9B59B6","#27AE60","#E74C3C","#F39C12","#1ABC9C"]
BG_COLOR  = "#F8F9FC"
GRID_COLOR = "#E0E4ED"
FONT      = "DejaVu Sans"

plt.rcParams.update({
    "figure.facecolor": BG_COLOR,
    "axes.facecolor": BG_COLOR,
    "axes.edgecolor": "#C5CAD6",
    "axes.labelcolor": "#2C3E50",
    "xtick.color": "#2C3E50",
    "ytick.color": "#2C3E50",
    "grid.color": GRID_COLOR,
    "grid.linestyle": "--",
    "grid.alpha": 0.7,
    "font.family": FONT,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
})

OUTPUT = "/mnt/user-data/outputs/"

In [80]:
df_raw = pd.read_csv("D:\Data Analytics\Project 7\globalb2bdataset.csv", encoding='latin-1')

print(f"\n📐 Shape          : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\n🗂  Column Types:\n{df_raw.dtypes.to_string()}")
print(f"\n📋 First 5 rows:\n{df_raw.head(5).to_string()}")
print(f"\n📋 Last 5 rows:\n{df_raw.tail(5).to_string()}")

print(f"\n🔍 Missing Values:")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(miss_df.to_string())

print(f"\n🔁 Duplicate Rows: {df_raw.duplicated().sum()}")


📐 Shape          : 763 rows × 5 columns

🗂  Column Types:
Decision Maker Name     object
Decision Maker Title    object
Industry                object
Email Address           object
Country                 object

📋 First 5 rows:
  Decision Maker Name                                        Decision Maker Title               Industry                  Email Address         Country
0            A Ansari                           Non Executive Director Co-Founder                    NaN        aj@turquoisehorizon.com   United States
1  A Ansmith Franklin                         Group Director Of Food And Beverage            Hospitality  ansmith.franklin@marriott.com           India
2    A Cameron Strong  Chief Executive Officer Of Invis And Mortgage Intelligence     Financial Services         cameronstrong@invis.ca          Canada
3  A Cwc Mark Wertman                                                       Owner        Events Services        cwcmark@qwestoffice.net  United Kingdom
4        

In [81]:
# Email validation
def valid_email(e):
    if pd.isna(e): return False
    return bool(re.match(r'^[\w.\-+]+@[\w\-]+\.[\w.\-]+$', str(e).strip()))

df_raw['_email_valid'] = df_raw['Email Address'].apply(valid_email)
print(f"\n📧 Invalid Emails : {(~df_raw['_email_valid']).sum()} / {len(df_raw)}")


📧 Invalid Emails : 0 / 763


In [82]:
# Unusual characters in names / titles
def has_special(s):
    if pd.isna(s): return False
    return bool(re.search(r'[^\w\s\-.,&/()\']', str(s)))
special_names  = df_raw['Decision Maker Name'].apply(has_special).sum()
special_titles = df_raw['Decision Maker Title'].apply(has_special).sum()
print(f"⚠️  Unusual chars in Names  : {special_names}")
print(f"⚠️  Unusual chars in Titles : {special_titles}")

⚠️  Unusual chars in Names  : 4
⚠️  Unusual chars in Titles : 2


In [83]:
# SECTION 2 — DATA CLEANING & PREPROCESSING

In [84]:
df = df_raw.copy()

In [85]:
# 2a. Remove duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"\n✅ Removed {before - len(df)} duplicate rows → {len(df):,} remain")


✅ Removed 0 duplicate rows → 763 remain


In [86]:
# 2b. Strip whitespace from all string columns
str_cols = df.select_dtypes(include='object').columns
for c in str_cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({'nan': np.nan, 'NaN': np.nan, '': np.nan})

In [87]:
# 2c. Standardize Country names
country_map = {
    'USA': 'United States', 'US': 'United States', 'U.S.': 'United States',
    'U.S.A.': 'United States', 'UK': 'United Kingdom', 'U.K.': 'United Kingdom',
    'Great Britain': 'United Kingdom', 'UAE': 'United Arab Emirates',
    'KSA': 'Saudi Arabia', 'S. Korea': 'South Korea',
}
df['Country'] = df['Country'].replace(country_map)
df['Country'] = df['Country'].str.title()

In [88]:
# 2d. Fill missing Industry/Country with 'Unknown'
df['Industry']  = df['Industry'].fillna('Unknown')
df['Country']   = df['Country'].fillna('Unknown')
df['Decision Maker Title'] = df['Decision Maker Title'].fillna('Unknown')

In [89]:
# 2e. Normalize industry names (title-case, strip extra spaces)
df['Industry'] = df['Industry'].str.title().str.replace(r'\s+', ' ', regex=True)

In [90]:
# 2f. Normalize titles
df['Decision Maker Title'] = (df['Decision Maker Title']
    .str.title()
    .str.replace(r'\s+', ' ', regex=True))

In [91]:
# 2g. Validate / clean emails
df['Email_Valid'] = df['Email Address'].apply(valid_email)
print(f"✅ Email valid after cleaning : {df['Email_Valid'].sum()} / {len(df)}")

# Drop the temp column used in section 1
df.drop(columns=['_email_valid'], errors='ignore', inplace=True)

print(f"\nFinal clean dataset shape: {df.shape}")

✅ Email valid after cleaning : 763 / 763

Final clean dataset shape: (763, 6)


In [92]:
# SECTION 3 — EXPLORATORY DATA ANALYSIS (EDA)

In [93]:
# ── 3a. Leadership role classification ───────────────────────────────────────
def classify_role(title):
    t = str(title).lower()
    if any(k in t for k in ['chief','ceo','coo','cfo','cto','cmo','cso','c-suite',
                             'president','executive vice','evp']):
        return 'C-Level'
    if 'founder' in t or 'co-founder' in t or 'owner' in t:
        return 'Founder / Owner'
    if 'director' in t:
        return 'Director'
    if 'manager' in t or 'head of' in t or 'head, ' in t:
        return 'Manager / Head'
    if any(k in t for k in ['vp','vice president','svp','evp']):
        return 'VP-Level'
    if any(k in t for k in ['partner','principal','associate']):
        return 'Partner / Principal'
    if any(k in t for k in ['consultant','advisor','specialist','analyst','engineer',
                             'architect','developer','scientist']):
        return 'Specialist / Consultant'
    return 'Other'

df['Role_Category'] = df['Decision Maker Title'].apply(classify_role)

role_counts = df['Role_Category'].value_counts()
print(f"\n📊 Role Category distribution:\n{role_counts.to_string()}")


📊 Role Category distribution:
Role_Category
Manager / Head             303
Other                      190
C-Level                    164
Specialist / Consultant     44
Founder / Owner             30
Partner / Principal         26
VP-Level                     6


In [94]:
# ── 3b. Industry counts ───────────────────────────────────────────────────────
industry_counts = df[df['Industry'] != 'Unknown']['Industry'].value_counts().head(20)
print(f"\n🏭 Top 20 Industries:\n{industry_counts.to_string()}")


🏭 Top 20 Industries:
Industry
Hospitality                            1
Financial Services                     1
Events Services                        1
Management Consulting                  1
Higher Education                       1
Renewables & Environment               1
Maritime                               1
Music                                  1
Education Management                   1
Secondary Education                    1
Think Tanks                            1
Accounting                             1
Investment Banking                     1
Consumer Goods                         1
Import And Export                      1
Textiles                               1
Electrical/Electronic Manufacturing    1
Photography                            1
Banking                                1
Business Supplies And Equipment        1


In [95]:
# ── 3c. Country counts ────────────────────────────────────────────────────────
country_counts = df[df['Country'] != 'Unknown']['Country'].value_counts().head(20)
print(f"\n🌍 Top 20 Countries:\n{country_counts.to_string()}")


🌍 Top 20 Countries:
Country
Australia         327
United States     317
India              48
Canada             36
United Kingdom     35


In [96]:
# SECTION 4 — B2B SALES INTELLIGENCE INSIGHTS (printed summary)

In [97]:
print(f"\n🎯 Top 5 Industries by Decision-Maker Count:")
print(industry_counts.head(5).to_string())

print(f"\n🌐 Top 5 Countries for B2B Targeting:")
print(country_counts.head(5).to_string())

print(f"\n👔 Most Frequent Leadership Roles:")
print(role_counts.to_string())

c_level_pct = round(role_counts.get('C-Level',0)/len(df)*100,1)
founder_pct  = round(role_counts.get('Founder / Owner',0)/len(df)*100,1)
print(f"\n💡 {c_level_pct}% are C-Level — direct budget authority")
print(f"💡 {founder_pct}% are Founders/Owners — ideal for startup-focused campaigns")


🎯 Top 5 Industries by Decision-Maker Count:
Industry
Hospitality              1
Financial Services       1
Events Services          1
Management Consulting    1
Higher Education         1

🌐 Top 5 Countries for B2B Targeting:
Country
Australia         327
United States     317
India              48
Canada             36
United Kingdom     35

👔 Most Frequent Leadership Roles:
Role_Category
Manager / Head             303
Other                      190
C-Level                    164
Specialist / Consultant     44
Founder / Owner             30
Partner / Principal         26
VP-Level                     6

💡 21.5% are C-Level — direct budget authority
💡 3.9% are Founders/Owners — ideal for startup-focused campaigns


In [98]:
# SECTION 5 — FEATURE ENGINEERING

In [99]:
# 5a. Seniority Level
def seniority(role):
    mapping = {'C-Level':'Executive','VP-Level':'Senior',
               'Founder / Owner':'Executive','Director':'Senior',
               'Manager / Head':'Mid-Level','Partner / Principal':'Senior',
               'Specialist / Consultant':'Junior','Other':'Unknown'}
    return mapping.get(role,'Unknown')

df['Seniority_Level'] = df['Role_Category'].apply(seniority)

In [100]:
# 5b. Email Domain
df['Email_Domain'] = df['Email Address'].apply(
    lambda e: str(e).split('@')[-1].lower().strip() if pd.notna(e) and '@' in str(e) else np.nan)
# Flag free / personal email providers
FREE_DOMAINS = {'gmail.com','yahoo.com','hotmail.com','outlook.com',
                'aol.com','icloud.com','protonmail.com','live.com'}
df['Email_Type'] = df['Email_Domain'].apply(
    lambda d: 'Personal' if d in FREE_DOMAINS else ('Business' if pd.notna(d) else 'Unknown'))

In [101]:
# 5c. Region from Country
region_map = {
    'United States':'North America','Canada':'North America','Mexico':'North America',
    'United Kingdom':'Europe','Germany':'Europe','France':'Europe','Netherlands':'Europe',
    'Spain':'Europe','Italy':'Europe','Sweden':'Europe','Norway':'Europe',
    'Denmark':'Europe','Finland':'Europe','Belgium':'Europe','Switzerland':'Europe',
    'Portugal':'Europe','Ireland':'Europe','Poland':'Europe','Austria':'Europe',
    'India':'Asia-Pacific','China':'Asia-Pacific','Japan':'Asia-Pacific',
    'Australia':'Asia-Pacific','Singapore':'Asia-Pacific','South Korea':'Asia-Pacific',
    'Hong Kong':'Asia-Pacific','New Zealand':'Asia-Pacific','Philippines':'Asia-Pacific',
    'Indonesia':'Asia-Pacific','Malaysia':'Asia-Pacific','Thailand':'Asia-Pacific',
    'Pakistan':'Asia-Pacific','Bangladesh':'Asia-Pacific','Sri Lanka':'Asia-Pacific',
    'United Arab Emirates':'Middle East & Africa','Saudi Arabia':'Middle East & Africa',
    'Israel':'Middle East & Africa','South Africa':'Middle East & Africa',
    'Nigeria':'Middle East & Africa','Kenya':'Middle East & Africa',
    'Egypt':'Middle East & Africa','Qatar':'Middle East & Africa',
    'Brazil':'Latin America','Argentina':'Latin America','Colombia':'Latin America',
    'Chile':'Latin America','Peru':'Latin America','Venezuela':'Latin America',
}
df['Region'] = df['Country'].map(region_map).fillna('Other')
print("\n✅ Feature Engineering Summary:")
print(f"  Seniority_Level : {df['Seniority_Level'].value_counts().to_dict()}")
print(f"  Email_Type      : {df['Email_Type'].value_counts().to_dict()}")
print(f"  Region          : {df['Region'].value_counts().to_dict()}")


✅ Feature Engineering Summary:
  Seniority_Level : {'Mid-Level': 303, 'Executive': 194, 'Unknown': 190, 'Junior': 44, 'Senior': 32}
  Email_Type      : {'Business': 761, 'Personal': 2}
  Region          : {'Asia-Pacific': 375, 'North America': 353, 'Europe': 35}


In [102]:
# SECTION 6 — ADVANCED ANALYTICS (Clustering & Segmentation)

In [103]:
# Encode categorical vars for clustering
le_industry = LabelEncoder()
le_role     = LabelEncoder()
le_region   = LabelEncoder()

df_cluster = df.copy()
df_cluster['ind_enc']    = le_industry.fit_transform(df_cluster['Industry'].fillna('Unknown'))
df_cluster['role_enc']   = le_role.fit_transform(df_cluster['Role_Category'])
df_cluster['region_enc'] = le_region.fit_transform(df_cluster['Region'])

X = df_cluster[['ind_enc','role_enc','region_enc']].values

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X)

In [104]:
# Label clusters with business names
cluster_profiles = df.groupby('Cluster').agg(
    Count=('Cluster','count'),
    Top_Role=('Role_Category', lambda x: x.value_counts().index[0]),
    Top_Industry=('Industry', lambda x: x.value_counts().index[0]),
    Top_Region=('Region', lambda x: x.value_counts().index[0])
).reset_index()

cluster_labels = {
    0: 'Enterprise Buyers',
    1: 'Startup Founders',
    2: 'Technical Leadership',
    3: 'Operations / Mid-Market',
    4: 'Emerging Market Leaders'
}

In [105]:
# Reassign based on dominant role
def label_cluster(row):
    r = row['Top_Role']
    if r in ('Founder / Owner',): return 'Startup Founders'
    if r in ('C-Level',): return 'Enterprise Buyers'
    if r in ('Specialist / Consultant',): return 'Technical Leadership'
    if r in ('Manager / Head',): return 'Operations / Mid-Market'
    return 'Emerging Market Leaders'

cluster_profiles['Segment_Name'] = cluster_profiles.apply(label_cluster, axis=1)

label_map = dict(zip(cluster_profiles['Cluster'], cluster_profiles['Segment_Name']))
df['Segment'] = df['Cluster'].map(label_map)

print("\n📦 Cluster Profiles:")
print(cluster_profiles.to_string(index=False))


📦 Cluster Profiles:
 Cluster  Count       Top_Role             Top_Industry    Top_Region            Segment_Name
       0    138 Manager / Head    Management Consulting  Asia-Pacific Operations / Mid-Market
       1    141 Manager / Head               Accounting  Asia-Pacific Operations / Mid-Market
       2    209 Manager / Head                  Unknown North America Operations / Mid-Market
       3    139 Manager / Head              Hospitality  Asia-Pacific Operations / Mid-Market
       4    136 Manager / Head Renewables & Environment  Asia-Pacific Operations / Mid-Market


In [106]:
# B2B Lead Segmentation Summary
print("\n🎯 B2B Lead Segments:")
print(df['Segment'].value_counts().to_string())


🎯 B2B Lead Segments:
Segment
Operations / Mid-Market    763


In [107]:
# SECTION 7 — VISUALIZATION DASHBOARD (saved as PNGs)

In [108]:
# ── FIG 1: Overview Dashboard (2x3 grid) ─────────────────────────────────────
fig = plt.figure(figsize=(22, 16), facecolor=BG_COLOR)
fig.suptitle("Global B2B Lead Intelligence — Overview Dashboard",
             fontsize=18, fontweight='bold', color='#1A3C5E', y=0.98)

gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

In [109]:
# 1. Top Industries (horizontal bar)
ax1 = fig.add_subplot(gs[0, :2])
top_ind = df[df['Industry']!='Unknown']['Industry'].value_counts().head(15)
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(top_ind))]
bars = ax1.barh(top_ind.index[::-1], top_ind.values[::-1], color=colors_bar[::-1],
                edgecolor='white', linewidth=0.5, height=0.7)
for bar, val in zip(bars, top_ind.values[::-1]):
    ax1.text(val + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val}', va='center', ha='left', fontsize=9, color='#2C3E50')
ax1.set_title("Top 15 Industries by Decision-Maker Count", fontweight='bold', pad=10)
ax1.set_xlabel("Count")
ax1.grid(axis='x', alpha=0.5)
ax1.spines[['top','right']].set_visible(False)

In [110]:
# 2. Role Category Pie
ax2 = fig.add_subplot(gs[0, 2])
role_data = df['Role_Category'].value_counts()
wedge_colors = PALETTE[:len(role_data)]
wedges, texts, autotexts = ax2.pie(
    role_data.values, labels=None, colors=wedge_colors,
    autopct='%1.1f%%', startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=1.5),
    pctdistance=0.82)
for at in autotexts:
    at.set_fontsize(8)
    at.set_color('white')
    at.set_fontweight('bold')
ax2.legend(role_data.index, loc='lower center', bbox_to_anchor=(0.5,-0.18),
           fontsize=8, ncol=2)
ax2.set_title("Decision-Maker Role Distribution", fontweight='bold', pad=10)

Text(0.5, 1.0, 'Decision-Maker Role Distribution')

In [111]:
# 3. Top Countries (bar)
ax3 = fig.add_subplot(gs[1, :2])
top_ctry = df[df['Country']!='Unknown']['Country'].value_counts().head(15)
bar_colors2 = [PALETTE[i % len(PALETTE)] for i in range(len(top_ctry))]
b2 = ax3.bar(range(len(top_ctry)), top_ctry.values, color=bar_colors2,
             edgecolor='white', linewidth=0.5, width=0.7)
for i, (bar, val) in enumerate(zip(b2, top_ctry.values)):
    ax3.text(i, val + 0.3, str(val), ha='center', va='bottom', fontsize=8.5, color='#2C3E50')
ax3.set_xticks(range(len(top_ctry)))
ax3.set_xticklabels(top_ctry.index, rotation=40, ha='right', fontsize=8.5)
ax3.set_title("Top 15 Countries by Decision-Maker Count", fontweight='bold', pad=10)
ax3.set_ylabel("Count")
ax3.grid(axis='y', alpha=0.5)
ax3.spines[['top','right']].set_visible(False)

In [112]:
# 4. Seniority + Region breakdown
ax4 = fig.add_subplot(gs[1, 2])
region_data = df['Region'].value_counts()
bar_colors3 = PALETTE[:len(region_data)]
ax4.barh(region_data.index[::-1], region_data.values[::-1],
         color=bar_colors3[::-1], edgecolor='white', linewidth=0.5, height=0.6)
for i, val in enumerate(region_data.values[::-1]):
    ax4.text(val + 0.2, i, str(val), va='center', fontsize=9, color='#2C3E50')
ax4.set_title("Decision-Makers by Region", fontweight='bold', pad=10)
ax4.set_xlabel("Count")
ax4.grid(axis='x', alpha=0.5)
ax4.spines[['top','right']].set_visible(False)

In [113]:
OUTPUT = r"D:\Data Analytics\Project 7\Output"
plt.savefig(f"{OUTPUT}/fig1_overview_dashboard.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig1_overview_dashboard.png")

✅ Saved: fig1_overview_dashboard.png


In [114]:
# ── FIG 2: Industry vs Role Heatmap ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 9), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)

top10_ind = df[df['Industry']!='Unknown']['Industry'].value_counts().head(10).index
heatmap_data = (df[df['Industry'].isin(top10_ind)]
                .groupby(['Industry','Role_Category'])
                .size().unstack(fill_value=0))

custom_cmap = LinearSegmentedColormap.from_list(
    'b2b', ['#EBF3FB','#2E86AB','#1A3C5E'], N=256)
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap=custom_cmap,
            linewidths=0.5, linecolor='white', ax=ax,
            annot_kws={"size": 10, "weight": "bold"},
            cbar_kws={"shrink": 0.6, "label": "Decision Maker Count"})

ax.set_title("Industry × Leadership Role Matrix\n(Top 10 Industries)",
             fontsize=15, fontweight='bold', color='#1A3C5E', pad=15)
ax.set_xlabel("Role Category", fontsize=11)
ax.set_ylabel("Industry", fontsize=11)
ax.tick_params(axis='x', rotation=35)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.savefig(f"{OUTPUT}/fig2_industry_role_heatmap.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig2_industry_role_heatmap.png")

✅ Saved: fig2_industry_role_heatmap.png


In [115]:
# ── FIG 3: Seniority & Email Intelligence ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor=BG_COLOR)
fig.suptitle("Decision-Maker Seniority & Email Intelligence",
             fontsize=15, fontweight='bold', color='#1A3C5E', y=1.01)

Text(0.5, 1.01, 'Decision-Maker Seniority & Email Intelligence')

In [116]:
# Seniority Level Donut
sen = df['Seniority_Level'].value_counts()
sen_colors = ['#1A3C5E','#2E86AB','#4ECDC4','#FF6B6B','#FFA500','#9B59B6']
wedges, texts, auto = axes[0].pie(
    sen.values, labels=None, colors=sen_colors[:len(sen)],
    autopct='%1.1f%%', startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2))
for at in auto:
    at.set_fontsize(9); at.set_color('white'); at.set_fontweight('bold')
axes[0].legend(sen.index, loc='lower center', bbox_to_anchor=(0.5,-0.15),
               fontsize=8.5, ncol=2)
axes[0].set_title("Seniority Level", fontweight='bold')

Text(0.5, 1.0, 'Seniority Level')

In [117]:
# Email Type
et = df['Email_Type'].value_counts()
et_colors = ['#2E86AB','#FF6B6B','#4ECDC4']
axes[1].bar(et.index, et.values, color=et_colors[:len(et)],
            edgecolor='white', linewidth=1.5, width=0.5)
for i, (idx, val) in enumerate(zip(et.index, et.values)):
    axes[1].text(i, val+1, str(val), ha='center', fontweight='bold', fontsize=11)
axes[1].set_title("Email Type (Business vs Personal)", fontweight='bold')
axes[1].set_ylabel("Count")
axes[1].grid(axis='y', alpha=0.5)
axes[1].spines[['top','right']].set_visible(False)

In [118]:
# Top Email Domains
top_domains = (df[df['Email_Type']=='Business']['Email_Domain']
               .value_counts().head(12))
axes[2].barh(top_domains.index[::-1], top_domains.values[::-1],
             color=PALETTE[1], edgecolor='white', linewidth=0.5, height=0.65)
for i, val in enumerate(top_domains.values[::-1]):
    axes[2].text(val+0.05, i, str(val), va='center', fontsize=8.5)
axes[2].set_title("Top 12 Business Email Domains", fontweight='bold')
axes[2].set_xlabel("Count")
axes[2].grid(axis='x', alpha=0.5)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()

In [119]:
plt.savefig(f"{OUTPUT}/fig3_seniority_email_intel.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig3_seniority_email_intel.png")

✅ Saved: fig3_seniority_email_intel.png


In [120]:
# ── FIG 4: B2B Clustering & Segmentation ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7), facecolor=BG_COLOR)
fig.suptitle("B2B Lead Clustering & Segmentation",
             fontsize=15, fontweight='bold', color='#1A3C5E', y=1.01)

Text(0.5, 1.01, 'B2B Lead Clustering & Segmentation')

In [121]:
# Segment distribution
seg_counts = df['Segment'].value_counts()
seg_colors = PALETTE[:len(seg_counts)]
axes[0].bar(range(len(seg_counts)), seg_counts.values,
            color=seg_colors, edgecolor='white', linewidth=1.5, width=0.6)
axes[0].set_xticks(range(len(seg_counts)))
axes[0].set_xticklabels(seg_counts.index, rotation=30, ha='right', fontsize=9)
for i, val in enumerate(seg_counts.values):
    axes[0].text(i, val+0.5, str(val), ha='center', fontsize=10, fontweight='bold')
axes[0].set_title("B2B Lead Segment Distribution", fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].grid(axis='y', alpha=0.5)
axes[0].spines[['top','right']].set_visible(False)

In [122]:
# PCA 2D scatter
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
scatter_colors = [PALETTE[c % len(PALETTE)] for c in df['Cluster']]
sc = axes[1].scatter(X_pca[:,0], X_pca[:,1],
                     c=[PALETTE[c % len(PALETTE)] for c in df['Cluster']],
                     alpha=0.65, s=50, edgecolors='white', linewidth=0.5)
for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    cx, cy = X_pca[mask, 0].mean(), X_pca[mask, 1].mean()
    axes[1].annotate(label_map[cluster_id], (cx, cy),
                     fontsize=8.5, fontweight='bold',
                     bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
axes[1].set_title("PCA Cluster Projection (2D)", fontweight='bold')
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
axes[1].grid(alpha=0.4)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()

In [123]:
plt.savefig(f"{OUTPUT}/fig4_clustering_segmentation.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig4_clustering_segmentation.png")

✅ Saved: fig4_clustering_segmentation.png


In [124]:
# ── FIG 5: Geographic & Industry Intelligence ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 13), facecolor=BG_COLOR)
fig.suptitle("Geographic & Industry Intelligence",
             fontsize=16, fontweight='bold', color='#1A3C5E', y=1.00)

Text(0.5, 1.0, 'Geographic & Industry Intelligence')

In [125]:
# Country × Region treemap-style horizontal bar (stacked by region)
top_ctry15 = df[df['Country']!='Unknown']['Country'].value_counts().head(12).index
ctry_reg = (df[df['Country'].isin(top_ctry15)]
            .groupby(['Country','Region'])
            .size().unstack(fill_value=0))
ctry_reg_sorted = ctry_reg.loc[ctry_reg.sum(axis=1).sort_values(ascending=True).index]
reg_colors_map = {
    'North America':'#1A3C5E','Europe':'#2E86AB','Asia-Pacific':'#4ECDC4',
    'Middle East & Africa':'#FF6B6B','Latin America':'#FFA500','Other':'#9B59B6'
}
bottom = np.zeros(len(ctry_reg_sorted))
for region in ctry_reg_sorted.columns:
    vals = ctry_reg_sorted[region].values
    axes[0,0].barh(ctry_reg_sorted.index, vals, left=bottom,
                   color=reg_colors_map.get(region,'#ccc'),
                   label=region, edgecolor='white', linewidth=0.5, height=0.7)
    bottom += vals
axes[0,0].set_title("Top 12 Countries (Stacked by Region)", fontweight='bold')
axes[0,0].set_xlabel("Decision-Maker Count")
axes[0,0].legend(loc='lower right', fontsize=8)
axes[0,0].grid(axis='x', alpha=0.5)
axes[0,0].spines[['top','right']].set_visible(False)

In [126]:
# Region × Role stacked bar
reg_role = df.groupby(['Region','Role_Category']).size().unstack(fill_value=0)
reg_role_pct = reg_role.div(reg_role.sum(axis=1), axis=0) * 100
role_palette = {r: PALETTE[i] for i,r in enumerate(reg_role_pct.columns)}
bottom2 = np.zeros(len(reg_role_pct))
for role in reg_role_pct.columns:
    axes[0,1].bar(reg_role_pct.index, reg_role_pct[role], bottom=bottom2,
                  label=role, color=role_palette.get(role,'#aaa'),
                  edgecolor='white', linewidth=0.5)
    bottom2 += reg_role_pct[role].values
axes[0,1].set_xticks(range(len(reg_role_pct)))
axes[0,1].set_xticklabels(reg_role_pct.index, rotation=30, ha='right', fontsize=9)
axes[0,1].set_title("Role Mix by Region (%)", fontweight='bold')
axes[0,1].set_ylabel("Percentage (%)")
axes[0,1].legend(loc='upper right', fontsize=7.5, ncol=2)
axes[0,1].grid(axis='y', alpha=0.4)
axes[0,1].spines[['top','right']].set_visible(False)

In [127]:
# Top industry breakdown by seniority (stacked)
top8_ind = df[df['Industry']!='Unknown']['Industry'].value_counts().head(8).index
ind_sen = (df[df['Industry'].isin(top8_ind)]
           .groupby(['Industry','Seniority_Level'])
           .size().unstack(fill_value=0))
ind_sen_sorted = ind_sen.loc[ind_sen.sum(axis=1).sort_values(ascending=True).index]
sen_colors_map = {
    'Executive':'#1A3C5E','Senior':'#2E86AB','Mid-Level':'#4ECDC4',
    'Junior':'#FFA500','Unknown':'#cccccc'
}
bottom3 = np.zeros(len(ind_sen_sorted))
for sen in ['Executive','Senior','Mid-Level','Junior','Unknown']:
    if sen in ind_sen_sorted.columns:
        vals = ind_sen_sorted[sen].values
        axes[1,0].barh(ind_sen_sorted.index, vals, left=bottom3,
                       color=sen_colors_map[sen], label=sen,
                       edgecolor='white', linewidth=0.5, height=0.65)
        bottom3 += vals
axes[1,0].set_title("Top 8 Industries — Seniority Breakdown", fontweight='bold')
axes[1,0].set_xlabel("Decision-Maker Count")
axes[1,0].legend(loc='lower right', fontsize=8.5)
axes[1,0].grid(axis='x', alpha=0.5)
axes[1,0].spines[['top','right']].set_visible(False)

In [128]:
# Email domain intelligence — business vs personal by region
et_reg = df.groupby(['Region','Email_Type']).size().unstack(fill_value=0)
et_reg_pct = et_reg.div(et_reg.sum(axis=1), axis=0) * 100
et_colors = {'Business':'#2E86AB','Personal':'#FF6B6B','Unknown':'#cccccc'}
bottom4 = np.zeros(len(et_reg_pct))
for et in ['Business','Personal','Unknown']:
    if et in et_reg_pct.columns:
        axes[1,1].bar(et_reg_pct.index, et_reg_pct[et], bottom=bottom4,
                      label=et, color=et_colors[et],
                      edgecolor='white', linewidth=0.5)
        bottom4 += et_reg_pct[et].values
axes[1,1].set_xticks(range(len(et_reg_pct)))
axes[1,1].set_xticklabels(et_reg_pct.index, rotation=30, ha='right', fontsize=9)
axes[1,1].set_title("Email Type by Region (%)", fontweight='bold')
axes[1,1].set_ylabel("Percentage (%)")
axes[1,1].legend(fontsize=9)
axes[1,1].grid(axis='y', alpha=0.4)
axes[1,1].spines[['top','right']].set_visible(False)

In [129]:
plt.tight_layout(h_pad=3.5, w_pad=3)
plt.savefig(f"{OUTPUT}/fig5_geo_industry_intelligence.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig5_geo_industry_intelligence.png")

✅ Saved: fig5_geo_industry_intelligence.png


In [130]:
# ── FIG 6: Lead Prioritization Matrix ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 9), facecolor=BG_COLOR)
ax.set_facecolor(BG_COLOR)

In [131]:
# Score each industry: executive_count × log(total_count)
ind_scores = []
for ind in df[df['Industry']!='Unknown']['Industry'].unique():
    sub = df[df['Industry']==ind]
    exec_count = (sub['Seniority_Level']=='Executive').sum()
    total = len(sub)
    exec_pct = exec_count / total * 100 if total > 0 else 0
    ind_scores.append({'Industry': ind, 'Total_DMs': total,
                       'Exec_Count': exec_count, 'Exec_Pct': exec_pct})

ind_df = pd.DataFrame(ind_scores).sort_values('Total_DMs', ascending=False).head(20)

scatter_sizes = ind_df['Total_DMs'] * 8
scatter_colors_map = ind_df['Exec_Pct'].values
sc = ax.scatter(ind_df['Total_DMs'], ind_df['Exec_Pct'],
                s=scatter_sizes, c=scatter_colors_map,
                cmap='YlOrRd', alpha=0.8, edgecolors='#1A3C5E', linewidth=1.5)

for _, row in ind_df.iterrows():
    ax.annotate(row['Industry'],
                (row['Total_DMs'], row['Exec_Pct']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, color='#2C3E50')

In [132]:
# Quadrant lines
med_x = ind_df['Total_DMs'].median()
med_y = ind_df['Exec_Pct'].median()
ax.axvline(med_x, color='#2E86AB', linestyle='--', alpha=0.6, linewidth=1.2)
ax.axhline(med_y, color='#FF6B6B', linestyle='--', alpha=0.6, linewidth=1.2)
ax.text(med_x+0.2, ax.get_ylim()[1]*0.97, 'High Volume →', fontsize=8, color='#2E86AB')
ax.text(ax.get_xlim()[0], med_y+0.5, 'High Executive %', fontsize=8, color='#FF6B6B', rotation=90)

cb = plt.colorbar(sc, ax=ax, shrink=0.7, label='Executive %')
ax.set_title("B2B Lead Prioritization Matrix\n(Bubble size = Total Decision-Makers)",
             fontsize=14, fontweight='bold', color='#1A3C5E', pad=15)
ax.set_xlabel("Total Decision-Makers in Industry", fontsize=11)
ax.set_ylabel("Executive-Level % in Industry", fontsize=11)
ax.grid(alpha=0.4)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()

In [133]:
plt.savefig(f"{OUTPUT}/fig6_lead_prioritization_matrix.png", dpi=150, bbox_inches='tight',
            facecolor=BG_COLOR)
plt.close()
print("✅ Saved: fig6_lead_prioritization_matrix.png")

✅ Saved: fig6_lead_prioritization_matrix.png
